# 04 — Time Series Analysis

**Role:** P3 — Time Series Analyst  
**Project:** Football Analytics Group Project  
**Goal:** Analyse team form and performance trends over time, then forecast the next 5–10 matchdays.

This notebook is designed to satisfy the time-series requirements in the project brief:
1. Load the master dataset from P1
2. Build matchday-level / weekly time series
3. Explore stationarity and autocorrelation
4. Fit and tune ARIMA / SARIMA
5. Optionally compare against Prophet
6. Forecast the next 5–10 matchdays
7. Evaluate forecast quality with MAE / RMSE
8. Produce figures and interpretations suitable for the report

> **Important:** This notebook assumes P1 has produced a clean master dataset.  
> If your column names differ, update the `COLUMN_MAP` section below.

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Optional model
PROPHET_AVAILABLE = True
try:
    from prophet import Prophet
except Exception:
    PROPHET_AVAILABLE = False

plt.rcParams["figure.figsize"] = (11, 5)
pd.set_option("display.max_columns", 200)

print("Prophet available:", PROPHET_AVAILABLE)

## 1. Load the master dataset

This section loads the cleaned match-level dataset created by P1.  
The purpose is to ensure that all downstream time-series analysis is based on the same engineered features and cleaned records used by the rest of the team.

In [ ]:
# Update this path if your project structure differs
DATA_PATH = Path("../data/clean/master_dataset.csv")

if not DATA_PATH.exists():
    print(f"File not found: {DATA_PATH.resolve()}")
    print("Update DATA_PATH so it points to P1's master dataset.")
else:
    df = pd.read_csv(DATA_PATH)
    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
    display(df.head())

## 2. Define the column mapping

Because different football datasets use different naming conventions, this notebook uses a flexible mapping layer.

Update the values below so they match your dataset.  
This makes the notebook reusable and prevents fragile hard-coded column names.

In [ ]:
COLUMN_MAP = {
    # Required columns
    "date": "date",
    "home_team": "home_team",
    "away_team": "away_team",
    "home_goals": "home_goals",
    "away_goals": "away_goals",

    # Optional columns you may already have from P1
    "season": "season",
    "league": "league",
    "matchday": "matchday",  # optional; if absent, matchday will be inferred
}

# Validate required columns
required = ["date", "home_team", "away_team", "home_goals", "away_goals"]
if "df" in globals():
    missing = [COLUMN_MAP[c] for c in required if COLUMN_MAP[c] not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in dataset: {missing}")

    data = df.copy()
    data[COLUMN_MAP["date"]] = pd.to_datetime(data[COLUMN_MAP["date"]])
    data = data.sort_values(COLUMN_MAP["date"]).reset_index(drop=True)
    display(data.head())

## 3. Transform match-level data into team-level time series

Time-series modelling requires one ordered series per team and metric.  
We convert each match into a team-centric record with:
- goals scored
- goals conceded
- points earned
- home/away indicator

This lets us analyse performance progression over time for each team.

In [ ]:
def build_team_match_table(data, cm):
    rows = []
    for _, r in data.iterrows():
        date = r[cm["date"]]
        home_team = r[cm["home_team"]]
        away_team = r[cm["away_team"]]
        hg = r[cm["home_goals"]]
        ag = r[cm["away_goals"]]

        # Home team view
        rows.append({
            "date": date,
            "team": home_team,
            "opponent": away_team,
            "venue": "Home",
            "goals_scored": hg,
            "goals_conceded": ag,
            "points": 3 if hg > ag else 1 if hg == ag else 0,
        })

        # Away team view
        rows.append({
            "date": date,
            "team": away_team,
            "opponent": home_team,
            "venue": "Away",
            "goals_scored": ag,
            "goals_conceded": hg,
            "points": 3 if ag > hg else 1 if hg == ag else 0,
        })

    team_matches = pd.DataFrame(rows).sort_values(["team", "date"]).reset_index(drop=True)
    team_matches["match_number"] = team_matches.groupby("team").cumcount() + 1
    return team_matches

if "data" in globals():
    team_matches = build_team_match_table(data, COLUMN_MAP)
    print("Team-match table shape:", team_matches.shape)
    display(team_matches.head(10))

## 4. Create rolling-form features

Rolling averages are useful because recent form often carries more predictive signal than full-season averages.  
Here we compute rolling 5-match averages for:
- points
- goals scored
- goals conceded

These features are also useful for integration with the ML notebook from P4.

In [ ]:
if "team_matches" in globals():
    team_matches["rolling_points_5"] = (
        team_matches.groupby("team")["points"]
        .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    )

    team_matches["rolling_goals_scored_5"] = (
        team_matches.groupby("team")["goals_scored"]
        .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    )

    team_matches["rolling_goals_conceded_5"] = (
        team_matches.groupby("team")["goals_conceded"]
        .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    )

    display(team_matches.head(10))

## 5. Aggregate to the time series of interest

The project brief asks for a weekly or matchday-by-matchday time series.  
This notebook uses **matchday-by-matchday** progression because:
- football league seasons have a natural sequential structure
- matchday series are easier to interpret in sporting terms
- the forecast horizon can be directly expressed as the next 5–10 matchdays

In [ ]:
def get_team_series(team_matches, team_name, metric="points"):
    ts = team_matches.loc[team_matches["team"] == team_name, ["date", "match_number", metric]].copy()
    ts = ts.rename(columns={metric: "y"}).sort_values("match_number")
    ts = ts.reset_index(drop=True)
    return ts

if "team_matches" in globals():
    available_teams = sorted(team_matches["team"].unique())
    print("Number of teams:", len(available_teams))
    print("Sample teams:", available_teams[:10])

## 6. Select the teams and metrics to model

You must model **at least 2 teams** and **at least 1 metric each**.  
A good justification is to select teams that are substantively interesting, for example:
- the league champion or title contender
- a mid-table team with volatile form
- a relegation-threatened team
- teams with contrasting attacking/defensive profiles

Update the selection below to match your league and project narrative.

In [ ]:
SELECTED_SERIES = [
    {"team": "Manchester City", "metric": "points"},
    {"team": "Arsenal", "metric": "goals_scored"},
]

if "team_matches" in globals():
    existing_teams = set(team_matches["team"].unique())
    for item in SELECTED_SERIES:
        if item["team"] not in existing_teams:
            print(f"Warning: '{item['team']}' not found in dataset. Replace it with a valid team name.")

### Team and metric selection rationale

**Write your justification here in prose before submission.**

Suggested wording:
- *Team A was selected because it was among the strongest teams in the league and provides a useful benchmark for consistent high performance.*
- *Team B was selected because its form was more volatile, making it useful for examining how well time-series models capture fluctuations.*
- *Points were chosen because they summarise match outcomes directly, while goals scored / conceded help separate attacking and defensive dynamics.*

## 7. Initial time-series exploration

Before modelling, we need to understand the data-generating process.  
This includes:
- plotting the series
- checking whether the mean and variance appear stable
- formally testing stationarity with the ADF test
- inspecting autocorrelation through ACF and PACF

These steps help justify differencing and the eventual ARIMA / SARIMA configuration.

In [ ]:
def adf_test(series, series_name="series"):
    result = adfuller(series.dropna(), autolag="AIC")
    output = {
        "series": series_name,
        "adf_statistic": result[0],
        "p_value": result[1],
        "n_lags": result[2],
        "n_obs": result[3],
    }
    return output

def plot_series_diagnostics(ts_df, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(ts_df["match_number"], ts_df["y"], marker="o")
    axes[0].set_title(f"{title} — Time Series")
    axes[0].set_xlabel("Match Number")
    axes[0].set_ylabel("Value")

    plot_acf(ts_df["y"].dropna(), ax=axes[1], lags=min(10, len(ts_df)-1))
    axes[1].set_title(f"{title} — ACF")

    max_pacf_lags = max(1, min(10, len(ts_df)//2 - 1))
    plot_pacf(ts_df["y"].dropna(), ax=axes[2], lags=max_pacf_lags, method="ywm")
    axes[2].set_title(f"{title} — PACF")

    plt.tight_layout()
    plt.show()

if "team_matches" in globals():
    for item in SELECTED_SERIES:
        ts = get_team_series(team_matches, item["team"], item["metric"])
        print(f"\n=== {item['team']} | {item['metric']} ===")
        display(ts.head())
        display(pd.DataFrame([adf_test(ts["y"], f"{item['team']} - {item['metric']}")]))
        plot_series_diagnostics(ts, f"{item['team']} - {item['metric']}")

### Interpretation guide for the exploration section

Use markdown after your plots to explain findings such as:
- whether the series appears stable or trend-like
- whether differencing seems necessary
- whether short-lag autocorrelation is present
- whether football context explains some variation (fixture difficulty, injuries, schedule congestion)

Do not just report the ADF p-value. Explain what it means for the modelling strategy.

## 8. Train-test split

To mimic real forecasting, we keep the last few matchdays as a holdout test set.  
This is better than random splitting because time-series evaluation must preserve chronology.

The default horizon below is 5 matchdays, which can be increased to 10 if your series is long enough.

In [ ]:
FORECAST_HORIZON = 5

def train_test_split_ts(ts_df, horizon=5):
    train = ts_df.iloc[:-horizon].copy()
    test = ts_df.iloc[-horizon:].copy()
    return train, test

if "team_matches" in globals():
    for item in SELECTED_SERIES:
        ts = get_team_series(team_matches, item["team"], item["metric"])
        train, test = train_test_split_ts(ts, FORECAST_HORIZON)
        print(f"{item['team']} | {item['metric']} -> train={len(train)}, test={len(test)}")

## 9. ARIMA / SARIMA modelling

The project brief explicitly says **do not just use defaults**.  
This section performs a small manual search over sensible `(p, d, q)` combinations and selects the best model by AIC.

Why this is acceptable:
- it is transparent and easy to explain
- it shows deliberate parameter choice
- it avoids blindly accepting a default configuration

You should describe what combinations you tried and why they were plausible based on the ACF/PACF and stationarity checks.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def tune_arima(train_series, p_values=(0,1,2), d_values=(0,1), q_values=(0,1,2), seasonal_order=None):
    results = []
    for p in p_values:
        for d in d_values:
            for q in q_values:
                order = (p, d, q)
                try:
                    model = SARIMAX(
                        train_series,
                        order=order,
                        seasonal_order=seasonal_order if seasonal_order else (0, 0, 0, 0),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    )
                    fitted = model.fit(disp=False)
                    results.append({
                        "order": order,
                        "seasonal_order": seasonal_order if seasonal_order else (0,0,0,0),
                        "aic": fitted.aic,
                        "bic": fitted.bic,
                    })
                except Exception:
                    continue
    results_df = pd.DataFrame(results).sort_values("aic").reset_index(drop=True)
    return results_df

def fit_best_arima(train_series, test_series, candidate_orders):
    evaluations = []
    for _, row in candidate_orders.iterrows():
        order = tuple(row["order"])
        seasonal_order = tuple(row["seasonal_order"])
        try:
            model = SARIMAX(
                train_series,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            fitted = model.fit(disp=False)
            forecast = fitted.get_forecast(steps=len(test_series))
            pred = forecast.predicted_mean
            mae = mean_absolute_error(test_series, pred)
            score_rmse = rmse(test_series, pred)
            evaluations.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "aic": fitted.aic,
                "mae": mae,
                "rmse": score_rmse,
                "model_obj": fitted,
            })
        except Exception:
            continue
    eval_df = pd.DataFrame(evaluations).sort_values(["rmse", "mae", "aic"]).reset_index(drop=True)
    return eval_df

all_model_results = {}

if "team_matches" in globals():
    for item in SELECTED_SERIES:
        ts = get_team_series(team_matches, item["team"], item["metric"])
        train, test = train_test_split_ts(ts, FORECAST_HORIZON)

        print(f"\n===== Tuning ARIMA for {item['team']} | {item['metric']} =====")
        candidates = tune_arima(train["y"], p_values=(0,1,2,3), d_values=(0,1), q_values=(0,1,2,3))
        display(candidates.head(10))

        best_eval = fit_best_arima(train["y"], test["y"], candidates.head(5))
        display(best_eval[["order", "seasonal_order", "aic", "mae", "rmse"]])

        all_model_results[(item["team"], item["metric"])] = {
            "ts": ts,
            "train": train,
            "test": test,
            "candidates": candidates,
            "best_eval": best_eval,
        }

### Notes on seasonal models

If your league data clearly shows seasonality or repeating cyclical behaviour, you can try a SARIMA model by adding a seasonal order such as `(P, D, Q, s)`.

For matchday-level football data, strong seasonality is often limited in a single season, so a non-seasonal ARIMA may be easier to justify unless your ACF suggests repeating structure.

## 10. Forecast and visualise actual vs. predicted values

This section plots:
- the training period
- the actual holdout values
- the forecast
- confidence intervals

This is the key visual that should go into the report and presentation.

In [ ]:
def plot_forecast(train, test, fitted_model, title):
    forecast_res = fitted_model.get_forecast(steps=len(test))
    pred_mean = forecast_res.predicted_mean
    conf_int = forecast_res.conf_int()

    plt.figure(figsize=(12, 5))
    plt.plot(train["match_number"], train["y"], label="Train", marker="o")
    plt.plot(test["match_number"], test["y"], label="Actual", marker="o")
    plt.plot(test["match_number"], pred_mean.values, label="Forecast", marker="o")
    plt.fill_between(
        test["match_number"],
        conf_int.iloc[:, 0].values,
        conf_int.iloc[:, 1].values,
        alpha=0.2,
        label="Confidence Interval"
    )
    plt.title(title)
    plt.xlabel("Match Number")
    plt.ylabel("Value")
    plt.legend()
    plt.show()

if all_model_results:
    for key, result in all_model_results.items():
        team, metric = key
        best_model = result["best_eval"].iloc[0]["model_obj"]
        plot_forecast(result["train"], result["test"], best_model, f"{team} — {metric}: Actual vs Forecast")

## 11. Generate a future 5–10 matchday forecast

Once the model is chosen, we can refit it on the full series and forecast the next unseen matchdays.  
This is the prospective forecast required by the assignment.

In [ ]:
FUTURE_HORIZON = 5

future_forecasts = {}

if all_model_results:
    for key, result in all_model_results.items():
        team, metric = key
        ts = result["ts"]
        best_row = result["best_eval"].iloc[0]
        best_order = tuple(best_row["order"])
        best_seasonal = tuple(best_row["seasonal_order"])

        final_model = SARIMAX(
            ts["y"],
            order=best_order,
            seasonal_order=best_seasonal,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)

        future = final_model.get_forecast(steps=FUTURE_HORIZON)
        pred = future.predicted_mean
        ci = future.conf_int()

        future_df = pd.DataFrame({
            "future_match_number": range(ts["match_number"].max() + 1, ts["match_number"].max() + FUTURE_HORIZON + 1),
            "forecast": pred.values,
            "lower_ci": ci.iloc[:, 0].values,
            "upper_ci": ci.iloc[:, 1].values,
        })

        future_forecasts[key] = future_df
        print(f"\nFuture forecast for {team} | {metric}")
        display(future_df)

## 12. Optional Prophet benchmark

Prophet can be useful when the series has trend changes or when you want an alternative modelling approach for comparison.

Because football matchday data is relatively short, Prophet may or may not outperform ARIMA.  
That comparison itself is analytically useful and should be documented.

In [ ]:
def run_prophet_benchmark(ts, horizon):
    prophet_df = ts[["date", "y"]].rename(columns={"date": "ds"}).copy()
    train = prophet_df.iloc[:-horizon]
    test = prophet_df.iloc[-horizon:]

    model = Prophet()
    model.fit(train)

    future = model.make_future_dataframe(periods=horizon, freq="W")
    forecast = model.predict(future)

    pred = forecast.tail(horizon)["yhat"].values
    mae = mean_absolute_error(test["y"], pred)
    score_rmse = rmse(test["y"], pred)

    return {
        "mae": mae,
        "rmse": score_rmse,
        "forecast_df": forecast,
        "test_df": test,
        "pred": pred,
    }

prophet_results = {}

if PROPHET_AVAILABLE and "team_matches" in globals():
    for item in SELECTED_SERIES:
        ts = get_team_series(team_matches, item["team"], item["metric"])
        try:
            res = run_prophet_benchmark(ts, FORECAST_HORIZON)
            prophet_results[(item["team"], item["metric"])] = res
            print(f"{item['team']} | {item['metric']} | Prophet RMSE = {res['rmse']:.3f}, MAE = {res['mae']:.3f}")
        except Exception as e:
            print(f"Prophet failed for {item['team']} | {item['metric']}: {e}")
else:
    print("Prophet is not installed or dataset not loaded. Skip this section if unavailable.")

## 13. Compare ARIMA and Prophet

This section is where you explicitly justify the final modelling choice.  
Do not just say one model had a lower error. Explain **why** that might be the case in football terms:
- stable teams may be easier to forecast
- highly volatile teams are harder to predict
- injuries, cup matches, squad rotation, and fixture congestion can cause shocks

In [ ]:
comparison_rows = []

if all_model_results:
    for key, result in all_model_results.items():
        team, metric = key
        arima_best = result["best_eval"].iloc[0]
        row = {
            "team": team,
            "metric": metric,
            "arima_mae": arima_best["mae"],
            "arima_rmse": arima_best["rmse"],
            "prophet_mae": np.nan,
            "prophet_rmse": np.nan,
            "preferred_model": "ARIMA",
        }
        if key in prophet_results:
            row["prophet_mae"] = prophet_results[key]["mae"]
            row["prophet_rmse"] = prophet_results[key]["rmse"]
            row["preferred_model"] = "Prophet" if row["prophet_rmse"] < row["arima_rmse"] else "ARIMA"
        comparison_rows.append(row)

if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)

## 14. Final interpretation and evaluation

For each selected team-metric pair, write a short interpretation covering:
1. whether the series was stationary or required differencing
2. which ARIMA order was selected and why
3. whether Prophet was tested and how it compared
4. the MAE / RMSE values
5. what football events may explain forecasting errors
6. what the forecast suggests for the next 5–10 matchdays

This is where you turn model output into football analysis.

### Suggested interpretation template

**Example structure:**

- *The points series for Team X showed moderate short-lag autocorrelation and required first differencing, which justified testing ARIMA models with d=1.*
- *Among the candidate models, ARIMA(p,d,q) achieved the lowest RMSE on the holdout period and was therefore selected as the final model.*
- *Prophet was also tested as an alternative benchmark but performed slightly worse, suggesting that the series did not contain strong trend changes that Prophet could exploit.*
- *Forecast errors were largest around unusually volatile matches, which may reflect injuries, fixture congestion, or opponent strength not fully captured in the univariate series.*
- *The final 5-matchday forecast suggests...*

## 15. Export results for the report

This section saves key tables for direct use in the report or slides.

In [ ]:
OUTPUT_DIR = Path("../data/clean/time_series_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if comparison_rows:
    comparison_df.to_csv(OUTPUT_DIR / "time_series_model_comparison.csv", index=False)

for key, forecast_df in future_forecasts.items():
    team, metric = key
    safe_team = team.lower().replace(" ", "_")
    forecast_df.to_csv(OUTPUT_DIR / f"forecast_{safe_team}_{metric}.csv", index=False)

print("Outputs saved to:", OUTPUT_DIR.resolve())

## 16. Final checklist for submission

Before submitting:
- [ ] Column names updated to match P1's dataset
- [ ] At least 2 teams modelled
- [ ] At least 1 metric per selected team
- [ ] ADF, ACF, and PACF included
- [ ] ARIMA tuning documented
- [ ] Prophet attempted or explicitly justified as not suitable / unavailable
- [ ] Forecast plot included with confidence intervals
- [ ] MAE and RMSE reported
- [ ] Markdown interpretation cells completed throughout the notebook
- [ ] Report methodology section copied into P5's final report